# Jina CLIP v2 멀티모달 임베딩 추출 파이프라인

##  개요
데스크테리어 상품 데이터(텍스트, 이미지)를 Jina CLIP v2 모델에 통과시켜 각각 1024차원의 벡터 데이터(`.npy`)로 변환하는 전처리 스크립트
추출된 파일은 이후 pgvector DB 적재 및 하이브리드 검색(Late Fusion)에 사용

## 주요 기능 및 버그 방어
1. **메타 텐서 에러 차단:** `accelerate` 충돌을 막기 위해 안정화된 구버전 라이브러리(transformers 4.40.1)를 강제 적용
2. **차원 붕괴 방지:** 스칼라 값으로 쪼개지는 현상을 막기 위해 모든 인코딩 입력값을 리스트(`[]`)로 래핑
3. **이미지 누락 방지:** `.zip` 압축 파일을 코랩 터미널 명령어로 일괄 해제

##  실행 순서
1. **환경 초기화:** 상단 메뉴에서 `런타임` -> `세션 다시 시작`을 클릭하여 꼬여있는 메모리 캐시를 비움
2. **셀 1 (환경 세팅) 실행:** 에러의 원인인 라이브러리를 지우고 필수 패키지를 세팅
3. **셀 2 (압축 해제) 실행:** 구글 드라이브 권한을 허용하고 `processed_images.zip` 파일 압축 풀기
4. **셀 3 (메인 로직) 실행:** 모델이 데이터를 벡터화하고 검증
5. **데이터 다운로드:** 실행 완료 후 터미널 맨 아래 출력된 결과가 `(데이터개수, 1024)`인지 확인. 정상이라면 구글 드라이브의 `embeddings_output` 폴더에 생성된 3개의 `.npy` 파일을 로컬 컴퓨터로 다운로드

In [1]:
!pip uninstall -y accelerate
!pip install -q transformers==4.40.1 torch Pillow numpy pandas tqdm einops

In [2]:
# 2. 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# 압축 파일 안에 있는 이미지들을 지정된 폴더로 풀기
!unzip -q -o '/content/drive/MyDrive/deskterior/processed_images.zip' -d '/content/drive/MyDrive/deskterior/processed_images'

print(" 압축 풀기 완료")

✅ 압축 풀기 완료! 이제 파이썬이 이미지를 볼 수 있습니다.


In [1]:
import os
import torch
import torch.nn.functional as F
from transformers import AutoModel
from PIL import Image
import pandas as pd
import numpy as np
from tqdm import tqdm

# =============================================================================
# 1. 경로 세팅
# =============================================================================
CSV_PATH = '/content/drive/MyDrive/deskterior/product.csv'
IMAGE_DIR = '/content/drive/MyDrive/deskterior/processed_images'
SAVE_DIR = '/content/drive/MyDrive/deskterior/embeddings_output'

os.makedirs(SAVE_DIR, exist_ok=True)

# =============================================================================
# 2. 모델 로딩 (순정 로딩)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[시스템] 사용 중인 디바이스: {device}")

print("[진행] Jina CLIP v2 모델 로딩 중...")
model = AutoModel.from_pretrained('jinaai/jina-clip-v2', trust_remote_code=True).to(device)
model.eval()

# =============================================================================
# 3. 데이터 로딩

df = pd.read_csv(CSV_PATH)
product_ids = []
text_embeddings = []
image_embeddings = []

print(f"\n[진행] 총 {len(df)}개 상품 분리 임베딩 시작...")

# =============================================================================
# 4. 임베딩 추출 메인 루프

with torch.no_grad():
    for index, row in tqdm(df.iterrows(), total=len(df)):
        p_id = str(row['id'])
        title = str(row.get('title', ''))
        category = str(row.get('category', 'ETC'))

        # A. 텍스트 임베딩 (리스트로 묶어서 차원 붕괴 방지)
        text_input = f"{title} - {category}"
        t_feat = model.encode_text([text_input])
        if isinstance(t_feat, np.ndarray):
            t_feat = torch.tensor(t_feat)
        t_vec = F.normalize(t_feat.to(device).float(), dim=-1)[0].cpu().numpy()

        # B. 이미지 임베딩 (리스트로 묶고, 에러 발생 시 원인 출력)
        img_path = os.path.join(IMAGE_DIR, f"{p_id}.png")
        if os.path.exists(img_path):
            try:
                img = Image.open(img_path).convert('RGB')
                i_feat = model.encode_image([img])
                if isinstance(i_feat, np.ndarray):
                    i_feat = torch.tensor(i_feat)
                i_vec = F.normalize(i_feat.to(device).float(), dim=-1)[0].cpu().numpy()
            except Exception as e:
                print(f"\n[이미지 오류] ID: {p_id} - {e}")
                i_vec = np.zeros(1024, dtype=np.float32)
        else:
            print(f"\n[경고] 이미지 누락 ID: {p_id}")
            i_vec = np.zeros(1024, dtype=np.float32)

        # C. 배열 저장
        product_ids.append(p_id)
        text_embeddings.append(t_vec)
        image_embeddings.append(i_vec)

# =============================================================================
# 5. 안전 검사 (Sanity Check) 및 파일 저장

print("\n[검사] 저장 전 데이터 차원(Shape)을 확인")

txt_arr = np.array(text_embeddings)
img_arr = np.array(image_embeddings)

print(f"[결과] Text Shape : {txt_arr.shape}")
print(f"[결과] Image Shape: {img_arr.shape}")

print("\n[저장] 데이터를 구글 드라이브에 기록")
np.save(os.path.join(SAVE_DIR, "product_ids.npy"), np.array(product_ids))
np.save(os.path.join(SAVE_DIR, "text_embeds.npy"), txt_arr)
np.save(os.path.join(SAVE_DIR, "image_embeds.npy"), img_arr)

print(f"\n[완료] 파일이 다음 경로에 저장되었습니다: {SAVE_DIR}")

KeyboardInterrupt: 